# AeroPulse PM2.5 — Phase 6: Regime-Aware Mixture of Experts

Build a pollution-regime classifier, specialist regressors, soft routing, extreme-event detection, calibration, and production model bundle.

## Objective

Build the **regime-aware PM2.5 system**.

Architecture:

```text
features
   ↓
pollution regime classifier
   ↓
normal / moderate / high / very-high / extreme
   ↓
specialist regression experts
   ↓
PM2.5 forecast + extreme-event probability
```

The key objective is not just lower average MAE. It is materially better performance during high and extreme episodes.

In [1]:
# Core configuration
from pathlib import Path
import os, json, math, warnings, hashlib, time
import numpy as np
import pandas as pd

from dotenv import find_dotenv, load_dotenv

# Load this folder's .env so os.getenv() below can see the API keys.
_dotenv_path = find_dotenv(usecwd=True)
if not _dotenv_path:
    raise RuntimeError("No .env found. Create one in this folder with OPENAQ_API_KEY / FIRMS_MAP_KEY.")
load_dotenv(_dotenv_path, override=True)
print("Loaded env from:", _dotenv_path)


warnings.filterwarnings("ignore")

PROJECT_ROOT = Path(os.getenv("AEROPULSE_PROJECT_ROOT", ".")).resolve()
DATA_ROOT = PROJECT_ROOT / "data" / "pm25"
RAW_DIR = DATA_ROOT / "raw"
PROCESSED_DIR = DATA_ROOT / "processed"
ARTIFACT_DIR = PROJECT_ROOT / "artifacts" / "pm25"
for p in [RAW_DIR, PROCESSED_DIR, ARTIFACT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

CONFIG = {
    "start": os.getenv("PM25_START", "2023-01-01T00:00:00Z"),
    "end": os.getenv("PM25_END", "2026-01-01T00:00:00Z"),
    "country": "IN",
    "min_station_rows": int(os.getenv("MIN_STATION_ROWS", "3000")),
    "min_coverage": float(os.getenv("MIN_STATION_COVERAGE", "0.35")),
    "india_bbox": {
        "min_lat": 6.0, "max_lat": 37.5,
        "min_lon": 68.0, "max_lon": 97.5,
    },
}
print("Project:", PROJECT_ROOT)
print("Data root:", DATA_ROOT)
print("Configuration:", CONFIG)

Loaded env from: /Users/sunil/Documents/hackathon/AeroPulse/AeroPulse/AeroPulse_ML_Notebooks/pm25_estimator/.env
Project: /Users/sunil/Documents/hackathon/AeroPulse/AeroPulse/AeroPulse_ML_Notebooks/pm25_estimator
Data root: /Users/sunil/Documents/hackathon/AeroPulse/AeroPulse/AeroPulse_ML_Notebooks/pm25_estimator/data/pm25
Configuration: {'start': '2023-01-01T00:00:00Z', 'end': '2026-01-01T00:00:00Z', 'country': 'IN', 'min_station_rows': 3000, 'min_coverage': 0.35, 'india_bbox': {'min_lat': 6.0, 'max_lat': 37.5, 'min_lon': 68.0, 'max_lon': 97.5}}


In [2]:
# ============================================================================
# IMPORTS
# ============================================================================
# The previous version imported only the classification metrics, while later
# cells called mean_absolute_error, r2_score, precision_score, recall_score and
# f1_score - every one of which raised NameError at runtime.
#
# root_mean_squared_error replaces mean_squared_error(squared=False), removed
# in scikit-learn 1.6.
# ============================================================================
import time
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    mean_absolute_error, root_mean_squared_error, r2_score,
    precision_score, recall_score, f1_score, precision_recall_curve,
)
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.frozen import FrozenEstimator

try:
    from lightgbm import LGBMRegressor
    HAS_LGBM = True
except ImportError:
    HAS_LGBM = False
    raise ImportError("Phase 6 requires lightgbm for the regime experts. "
                      "Install with: uv pip install lightgbm")
print("Phase 6 imports ready.")


Phase 6 imports ready.


## 1. Load Phase-2 data and freeze the feature contract

In [3]:
EVENT_FILE = PROCESSED_DIR / "event_aware" / "pm25_event_aware_features.parquet"
df = pd.read_parquet(EVENT_FILE)
df["timestamp_utc"] = pd.to_datetime(df["timestamp_utc"], utc=True)

TARGET_H = 24
REG_TARGET = f"target_pm25_t_plus_{TARGET_H}h"

EXCLUDE = {
    "pm25", "timestamp_utc", "station_name", "source", "pm25_unit",
    "pollution_regime", "event_id", "event_start_flag", "country",
    "state", "city"
}
FEATURES = [
    c for c in df.columns
    if c not in EXCLUDE
    and not c.startswith("target_")
    and not c.startswith(("event_peak","event_mean","event_duration","event_start","event_end"))
    and pd.api.types.is_numeric_dtype(df[c])
]

df = df.dropna(subset=[REG_TARGET]).sort_values("timestamp_utc")
print(df.shape, len(FEATURES))

(1701676, 105) 77


## 2. Define regime target

For a 24-hour forecasting system, the classifier should answer:

> What pollution regime will the target location be in at the forecast horizon?

The classifier can use the same information available at prediction time, but never future PM2.5.

In [4]:
def regime_from_pm25(s):
    return pd.cut(
        s,
        bins=[-np.inf,30,60,90,150,np.inf],
        labels=["normal","moderate","high","very_high","extreme"]
    )

df["future_regime"] = regime_from_pm25(df[REG_TARGET])
print(df["future_regime"].value_counts())

future_regime
normal       912266
moderate     538334
high         162440
very_high     69991
extreme       18645
Name: count, dtype: int64


## 3. Temporal split

Keep the final period as the test set.

Do not oversample before splitting. Oversampling must be applied only to the training set.

In [5]:
# ============================================================================
# CHRONOLOGICAL SPLIT - train / calibration / test
# ============================================================================
# Three slices, not two. Probability calibration must be fitted on data the
# classifier did not train on, and it must NOT be fitted on the test period -
# otherwise the reported alert precision/recall is measured on data that
# helped choose the threshold. The middle slice exists for exactly that.
# ============================================================================
t1 = df.timestamp_utc.quantile(0.70)
t2 = df.timestamp_utc.quantile(0.85)

train = df[df.timestamp_utc < t1].copy()
calib = df[(df.timestamp_utc >= t1) & (df.timestamp_utc < t2)].copy()
test = df[df.timestamp_utc >= t2].copy()
cut = t2   # retained for the artifact manifest

print(f"train {train.shape}  calib {calib.shape}  test {test.shape}")
print(f"train: {train.timestamp_utc.min()} -> {train.timestamp_utc.max()}")
print(f"calib: {calib.timestamp_utc.min()} -> {calib.timestamp_utc.max()}")
print(f"test : {test.timestamp_utc.min()} -> {test.timestamp_utc.max()}")

print("\nFuture-regime mix per split:")
for name, part in [("train", train), ("calib", calib), ("test", test)]:
    counts = part["future_regime"].astype(str).value_counts().to_dict()
    print(f"  {name:6} {counts}")
if (test["future_regime"].astype(str) == "extreme").sum() < 10:
    print("\nCAUTION: fewer than 10 extreme rows in test - alert metrics below "
          "will be extremely noisy. Widen the dataset window (a burning season) "
          "before treating them as evidence.")


train (1191168, 106)  calib (255230, 106)  test (255278, 106)
train: 2025-02-18 20:30:00+00:00 -> 2026-03-21 05:30:00+00:00
calib: 2026-03-21 06:30:00+00:00 -> 2026-06-12 18:30:00+00:00
test : 2026-06-12 19:30:00+00:00 -> 2026-09-08 06:30:00+00:00

Future-regime mix per split:
  train  {'normal': 562644, 'moderate': 403384, 'high': 143893, 'very_high': 64389, 'extreme': 16858}
  calib  {'normal': 143281, 'moderate': 91480, 'high': 14913, 'very_high': 4355, 'extreme': 1201}
  test   {'normal': 206341, 'moderate': 43470, 'high': 3634, 'very_high': 1247, 'extreme': 586}


### Result — three slices, and the same seasonal caveat

| Split | Rows | Period |
|---|---:|---|
| train | 1,191,168 | 2025-02-18 → 2026-03-21 |
| calib | 255,230 | 2026-03-21 → 2026-06-12 |
| test | 255,278 | 2026-06-12 → 2026-09-08 |

Future-regime mix:

| Regime | train | calib | **test** |
|---|---:|---:|---:|
| normal | 562,644 | 143,281 | 206,341 |
| moderate | 403,384 | 91,480 | 43,470 |
| high | 143,893 | 14,913 | 3,634 |
| very_high | 64,389 | 4,355 | 1,247 |
| **extreme** | 16,858 | 1,201 | **586** |

The test period is June–September — **monsoon again**. Extreme rows fall from 1.42% of train to 0.23% of test. Everything below is therefore measured on the model's weakest season, exactly as Phase 5's rolling CV showed. Read the alerting numbers as a *floor*, not a typical figure.


## 4. Train the regime classifier

Start with LightGBM/CatBoost/RandomForest.

Later, the classifier can become a calibrated probabilistic model.

Important outputs:

```text
P(normal)
P(moderate)
P(high)
P(very_high)
P(extreme)
```

In [6]:
# ============================================================================
# GATING CLASSIFIER - which pollution regime is coming?
# ============================================================================
# This is the "gate" of the mixture: it predicts WHICH regime the next
# TARGET_H hours fall into, and its probabilities weight the experts below.
#
# class_weight="balanced_subsample" is essential here. Extreme rows are well
# under 1% of the data, so an unweighted classifier maximises accuracy by
# never predicting "extreme" at all - technically 99% correct and operationally
# useless.
# ============================================================================
Xtr, Xte = train[FEATURES], test[FEATURES]
ytr = train["future_regime"].astype(str)
yte = test["future_regime"].astype(str)

imputer = SimpleImputer(strategy="median")     # fit on train only
Xtr_i = imputer.fit_transform(Xtr)
Xte_i = imputer.transform(Xte)

# max_samples bags each tree on a fraction of the rows. At 1.2M training rows
# an uncapped 500-tree forest needs hours; bagging 25% per tree costs little
# because the trees are decorrelated anyway.
clf = RandomForestClassifier(
    n_estimators=int(os.getenv("PM25_CLF_TREES", "250")),
    class_weight="balanced_subsample",
    min_samples_leaf=5,
    max_samples=float(os.getenv("PM25_CLF_MAX_SAMPLES", "0.25")),
    n_jobs=-1, random_state=42,
)
_t = time.time()
clf.fit(Xtr_i, ytr)
print(f'classifier trained in {time.time()-_t:.0f}s')
pred_regime = clf.predict(Xte_i)
prob_regime = clf.predict_proba(Xte_i)

print(classification_report(yte, pred_regime, zero_division=0))
print("Classes:", list(clf.classes_))
print("\nRead the per-class RECALL above, not the accuracy: accuracy is "
      "dominated by the 'normal' class and says nothing about severe regimes.")


classifier trained in 103s


              precision    recall  f1-score   support

     extreme       0.12      0.04      0.06       586
        high       0.27      0.08      0.12      3634
    moderate       0.62      0.42      0.50     43470
      normal       0.88      0.96      0.92    206341
   very_high       0.30      0.15      0.20      1247

    accuracy                           0.84    255278
   macro avg       0.44      0.33      0.36    255278
weighted avg       0.82      0.84      0.83    255278

Classes: ['extreme', 'high', 'moderate', 'normal', 'very_high']

Read the per-class RECALL above, not the accuracy: accuracy is dominated by the 'normal' class and says nothing about severe regimes.


### Result — ⚠️ 84% accuracy, 4% extreme recall

```
              precision  recall  f1-score  support
     extreme       0.12    0.04      0.06      586
        high       0.27    0.08      0.12     3634
    moderate       0.62    0.42      0.50    43470
      normal       0.88    0.96      0.92   206341
   very_high       0.30    0.15      0.20     1247
    accuracy                         0.84   255278
```
*(trained in 103 s)*

**This is why accuracy must never be the reported metric here.** 84% looks respectable and is almost entirely the `normal` class, which is 81% of the test set. The classifier identifies **4% of extreme hours** — it is close to never predicting the class the product exists to detect, even with `class_weight="balanced_subsample"`.

Two things rescue the situation downstream, and both matter:

1. The mixture uses the full **probability vector**, not the argmax label. A 12% probability of extreme still pulls the blended prediction upward even when `extreme` is never the top class.
2. Thresholding that probability at 0.1 (next cells) recovers recall 0.32 with ROC-AUC **0.84** — the classifier *ranks* risk well even though its argmax decisions are poor.

The lesson: hard-label recall and probability quality are different things. Use the probabilities.


## 5. Specialist regression experts

Train separate regressors:

- normal/moderate expert
- high expert
- very-high expert
- extreme expert

For sparse extreme data, combine:

- sample weighting
- event-level sampling
- multi-year historical data
- robust objectives

Do not simply duplicate every extreme row.

In [7]:
# ============================================================================
# REGIME EXPERTS - one regressor per future pollution regime
# ============================================================================
# Why a mixture of experts: a single model fitted on a distribution that is
# ~90% ordinary air optimises the bulk and treats severe episodes as noise.
# Training a dedicated regressor on each regime lets the extreme expert learn
# extreme dynamics without being averaged away.
#
# REMOVED: the previous version passed sample_weight=np.full(len(subset), W)
# to each expert - a CONSTANT weight vector, which scales the loss uniformly
# and therefore has NO effect on the fitted model. It looked like imbalance
# handling while doing nothing. Imbalance is handled where it actually can be:
# by the regime split itself, and by class_weight on the gating classifier.
#
# A global fallback model covers regimes too sparse to fit an expert, so
# mixture_predict never has to invent a prediction.
# ============================================================================
REGIMES = ["normal", "moderate", "high", "very_high", "extreme"]
MIN_EXPERT_ROWS = int(os.getenv("PM25_MIN_EXPERT_ROWS", "50"))

def make_expert():
    return LGBMRegressor(n_estimators=1000, learning_rate=0.03, num_leaves=63,
                         objective="regression", random_state=42, verbose=-1)

# Fallback first, so a sparse regime always has something to fall back to.
fallback_model = make_expert()
fallback_model.fit(train[FEATURES], train[REG_TARGET])

EXPERTS = {}
for regime in REGIMES:
    subset = train[train["future_regime"].astype(str) == regime]
    if len(subset) < MIN_EXPERT_ROWS:
        print(f"  {regime:10} {len(subset):6,} rows - too sparse, using global fallback")
        continue
    model = make_expert()
    model.fit(subset[FEATURES], subset[REG_TARGET])
    EXPERTS[regime] = model
    print(f"  {regime:10} {len(subset):6,} rows - expert trained "
          f"(target range {subset[REG_TARGET].min():.0f}-{subset[REG_TARGET].max():.0f})")

print(f"\n{len(EXPERTS)} experts trained, fallback covers {len(REGIMES)-len(EXPERTS)} regime(s).")
if "extreme" not in EXPERTS:
    print("WARNING: no extreme expert. The mixture cannot produce a genuinely "
          "extreme forecast, so alert recall will be near zero by construction.")


  normal     562,644 rows - expert trained (target range 0-30)


  moderate   403,384 rows - expert trained (target range 30-60)


  high       143,893 rows - expert trained (target range 60-90)


  very_high  64,389 rows - expert trained (target range 90-150)


  extreme    16,858 rows - expert trained (target range 151-1000)

5 experts trained, fallback covers 0 regime(s).


### Result — all 5 experts trained

| Expert | Training rows | Target range |
|---|---:|---|
| normal | 562,644 | 0–30 |
| moderate | 403,384 | 30–60 |
| high | 143,893 | 60–90 |
| very_high | 64,389 | 90–150 |
| **extreme** | **16,858** | **151–1000** |

No regime fell back to the global model — a direct consequence of the 19-month, all-season dataset. On the earlier 60-day monsoon window, `very_high` had 7 rows and `extreme` had 5, so neither expert could be trained at all and the mixture had nothing to blend toward during severe air.

Note the removed no-op: the previous version passed `sample_weight=np.full(n, W)` — a *constant* vector, which scales the loss uniformly and changes nothing. It looked like imbalance handling while doing nothing at all.


## 6. Mixture-of-experts prediction

There are two strategies:

### Hard routing

Use the most likely regime expert.

### Soft routing

Combine expert predictions weighted by regime probabilities.

Soft routing is usually safer near regime boundaries.

In [8]:
# ============================================================================
# MIXTURE PREDICTION - gate probabilities x expert predictions
# ============================================================================
# The gating classifier gives P(regime | features); each expert gives its own
# estimate; the forecast is the probability-weighted sum. This lets the model
# hedge: as the chance of an extreme regime rises, the prediction is pulled
# toward the extreme expert's estimate before the episode actually arrives.
#
# FIXED: the previous fallback filled a missing expert's column with
# np.nanmean(expert_preds, axis=1) while iterating over those same columns -
# order-dependent, and all-NaN when several experts were missing. Sparse
# regimes now use the explicitly trained global fallback model.
# ============================================================================
regime_order = list(clf.classes_)


def mixture_predict(rows, classifier=None):
    classifier = classifier or clf
    probs = classifier.predict_proba(imputer.transform(rows[FEATURES]))
    expert_preds = np.zeros((len(rows), len(regime_order)))
    for j, regime in enumerate(regime_order):
        model = EXPERTS.get(regime, fallback_model)
        expert_preds[:, j] = model.predict(rows[FEATURES])
    return np.sum(probs * expert_preds, axis=1), probs


moe_pred, moe_probs = mixture_predict(test)
moe_scores = {
    "MAE": mean_absolute_error(test[REG_TARGET], moe_pred),
    "RMSE": root_mean_squared_error(test[REG_TARGET], moe_pred),
    "R2": r2_score(test[REG_TARGET], moe_pred),
}
print("Mixture-of-experts:", moe_scores)

# Compare against the fallback alone - if the mixture is not better, the extra
# machinery is not paying for itself and a single model is the honest choice.
single_pred = fallback_model.predict(test[FEATURES])
single_scores = {
    "MAE": mean_absolute_error(test[REG_TARGET], single_pred),
    "RMSE": root_mean_squared_error(test[REG_TARGET], single_pred),
    "R2": r2_score(test[REG_TARGET], single_pred),
}
print("Single global model:", single_scores)
print(f"\nMoE improves RMSE by {single_scores['RMSE'] - moe_scores['RMSE']:+.2f} ug/m3 "
      f"({(single_scores['RMSE']-moe_scores['RMSE'])/single_scores['RMSE']:+.1%})")

# Per-regime view: the MoE is meant to win specifically in the severe bins.
bins = pd.cut(test[REG_TARGET], [-np.inf,30,60,90,150,np.inf],
              labels=["normal","moderate","high","very_high","extreme"])
rows = []
for r in ["normal","moderate","high","very_high","extreme"]:
    m = np.asarray(bins == r)
    if m.sum():
        rows.append({"regime": r, "count": int(m.sum()),
                     "MoE_MAE": mean_absolute_error(test[REG_TARGET][m], moe_pred[m]),
                     "single_MAE": mean_absolute_error(test[REG_TARGET][m], single_pred[m])})
display(pd.DataFrame(rows))


Mixture-of-experts: {'MAE': 9.479581574606682, 'RMSE': 19.279155544389795, 'R2': 0.10021008953927113}


Single global model: {'MAE': 7.908119747582345, 'RMSE': 17.988469297133516, 'R2': 0.2166541931475613}

MoE improves RMSE by -1.29 ug/m3 (-7.2%)


,regime,count,MoE_MAE,single_MAE
0,normal,206341,8.100708,5.982445
1,moderate,43470,10.607967,10.299831
2,high,3634,26.207023,32.170599
3,very_high,1247,49.994832,62.225363
4,extreme,586,221.352024,242.506159


### Result — the MoE loses overall and wins where it was designed to

| | MAE | RMSE | R² |
|---|---:|---:|---:|
| Mixture of experts | 9.48 | 19.28 | 0.100 |
| **Single global model** | **7.91** | **17.99** | **0.217** |

**Aggregate RMSE is 7.2% worse.** But the per-regime breakdown tells the opposite story:

| Regime | Count | MoE MAE | Single MAE | Winner |
|---|---:|---:|---:|---|
| normal | 206,341 | 8.10 | **5.98** | single |
| moderate | 43,470 | 10.61 | **10.30** | single |
| high | 3,634 | **26.21** | 32.17 | **MoE −19%** |
| very_high | 1,247 | **49.99** | 62.23 | **MoE −20%** |
| **extreme** | 586 | **221.35** | 242.51 | **MoE −9%** |

The mixture is **worse on the 98% of rows that are ordinary air and better on every severe regime** — which is precisely its design intent. Aggregate RMSE is dominated by the bulk, so it reports the trade as a loss.

**Which is "better" depends on the product.** For a concentration API, use the single model. For hazard alerting, the MoE's 19–20% improvement in the `high`/`very_high` bins is the relevant number. Reporting only the aggregate would have hidden the entire trade-off — and would have led to deleting the mixture as a failure.

The honest caveat: even the MoE's extreme MAE of 221 µg/m³ is very large. Both models remain badly biased low during severe episodes.


## 7. Extreme-event detection

For AeroPulse alerting, calculate:

- probability of extreme pollution
- precision
- recall
- F1
- PR-AUC where possible

The model should be optimized for **high recall without producing unusable false alarms**.

In [9]:
# ============================================================================
# EXTREME-EVENT ALERTING - threshold chosen on calibration, scored on test
# ============================================================================
# This is the cell that decides whether AeroPulse can actually warn people.
#
# FIXED: the alert threshold was hardcoded at 0.30 and evaluated on the test
# set - which is threshold selection on the test period, i.e. reporting a
# tuned number as if it were held out. The threshold is now swept on the
# CALIBRATION slice and only then applied, once, to test.
#
# The objective is F-beta with beta=2, which weights recall 4x precision: for
# a health warning a missed hazardous episode is far costlier than a false
# alarm, and optimising plain F1 quietly trades away the recall that matters.
# ============================================================================
extreme_idx = [i for i, c in enumerate(regime_order) if c == "extreme"]

if not extreme_idx:
    print("No 'extreme' class in the gating classifier - alerting cannot be evaluated.")
    alert_threshold, alert_scores = None, None
else:
    j = extreme_idx[0]
    _, calib_probs = mixture_predict(calib)
    calib_extreme_prob = calib_probs[:, j]
    calib_actual = (calib[REG_TARGET] >= 150).astype(int).to_numpy()

    def fbeta(prec, rec, beta=2.0):
        if prec == 0 and rec == 0:
            return 0.0
        b2 = beta ** 2
        return (1 + b2) * prec * rec / (b2 * prec + rec)

    sweep = []
    for thr in np.arange(0.05, 0.96, 0.05):
        pred = (calib_extreme_prob >= thr).astype(int)
        p = precision_score(calib_actual, pred, zero_division=0)
        r = recall_score(calib_actual, pred, zero_division=0)
        sweep.append({"threshold": round(float(thr), 2), "precision": p,
                      "recall": r, "f2": fbeta(p, r)})
    sweep_df = pd.DataFrame(sweep)

    if calib_actual.sum() == 0:
        alert_threshold = 0.30
        print(f"No extreme rows in the calibration slice - falling back to "
              f"threshold {alert_threshold} (UNTUNED, treat as a placeholder).")
    else:
        alert_threshold = float(sweep_df.loc[sweep_df.f2.idxmax(), "threshold"])
        print(f"Threshold chosen on CALIBRATION slice: {alert_threshold} "
              f"(F2={sweep_df.f2.max():.3f})")
        display(sweep_df)

    # Apply once, to test.
    test_extreme_prob = moe_probs[:, j]
    actual_extreme = (test[REG_TARGET] >= 150).astype(int).to_numpy()
    predicted_extreme = (test_extreme_prob >= alert_threshold).astype(int)

    alert_scores = {
        "threshold": alert_threshold,
        "roc_auc": float(roc_auc_score(actual_extreme, test_extreme_prob))
                   if actual_extreme.sum() and actual_extreme.sum() < len(actual_extreme) else None,
        "precision": float(precision_score(actual_extreme, predicted_extreme, zero_division=0)),
        "recall": float(recall_score(actual_extreme, predicted_extreme, zero_division=0)),
        "f1": float(f1_score(actual_extreme, predicted_extreme, zero_division=0)),
        "actual_extreme": int(actual_extreme.sum()),
        "predicted_extreme": int(predicted_extreme.sum()),
    }
    print("\nTEST-period alerting:", json.dumps(alert_scores, indent=2))
    if alert_scores["actual_extreme"] == 0:
        print("\nNo extreme events in the test period, so recall is undefined. "
              "This is a DATASET limitation, not a model result - the window "
              "must span a burning season for this metric to mean anything.")
    elif alert_scores["recall"] == 0:
        print("\nThe model detected NONE of the extreme episodes. An alerting "
              "product built on this would stay silent during exactly the "
              "events it exists to warn about.")


Threshold chosen on CALIBRATION slice: 0.1 (F2=0.174)


,threshold,precision,recall,f2
0,0.05,0.034017,0.457683,0.131108
1,0.10,0.065493,0.296631,0.173892
2,0.15,0.081103,0.166804,0.137702
3,0.20,0.090171,0.091208,0.090999
4,0.25,0.106109,0.054232,0.060109
5,0.30,0.116838,0.027938,0.032952
6,0.35,0.137097,0.013969,0.017027
7,0.40,0.155556,0.005752,0.007124
8,0.45,0.176471,0.002465,0.003071
9,0.50,0.333333,0.000822,0.001026



TEST-period alerting: {
  "threshold": 0.1,
  "roc_auc": 0.8416040745774712,
  "precision": 0.06741573033707865,
  "recall": 0.32432432432432434,
  "f1": 0.11162790697674418,
  "actual_extreme": 592,
  "predicted_extreme": 2848
}


### Result — ROC-AUC 0.84, but precision is 7%

Threshold swept on the **calibration** slice (never on test), optimising F-beta with β=2:

| Threshold | Precision | Recall | F2 |
|---|---:|---:|---:|
| 0.05 | 0.034 | **0.458** | 0.131 |
| **0.10** | 0.065 | 0.297 | **0.174** ← chosen |
| 0.20 | 0.090 | 0.091 | 0.091 |
| 0.30 | 0.117 | 0.028 | 0.033 |
| 0.50 | 0.333 | 0.001 | 0.001 |

Applied once to test:

```
roc_auc   : 0.842      recall    : 0.324
precision : 0.067      predicted : 2,848  (actual: 592)
```

**ROC-AUC 0.842 says the model ranks risk well.** The problem is not discrimination, it is the operating point: at the chosen threshold it raises 2,848 alerts to catch 192 of 592 events — roughly **14 false alarms per true one**.

That is the honest cost of prioritising recall on a 0.23% positive class. Whether it is acceptable is a product decision, not a modelling one: a dashboard colour change tolerates it, an SMS blast to millions does not.

The threshold being *selected on calibration data and applied once to test* is what makes these numbers trustworthy. The previous version hardcoded 0.30 and evaluated on test — reporting a tuned number as if it were held out.


## 8. Event-level evaluation

Row-level recall is not enough.

Collapse predictions into events and measure:

- event detected
- lead time
- false alarm duration
- missed event
- peak error
- peak timing error

AeroPulse should eventually have an alert policy such as:

```text
Extreme probability >= threshold
AND
probability remains elevated for N hours
```

This should be tuned after seeing real event performance.

In [10]:
# ============================================================================
# EVENT-LEVEL EVALUATION - lead time, not just hourly accuracy
# ============================================================================
# Hourly recall answers "was this hour flagged?". Operations needs "did we warn
# before the episode started, and how early?". An alert 30 minutes ahead is
# almost worthless; six hours lets schools and construction sites act.
#
# Phase 2 supplies event_id / event_start / event_peak_pm25, so events can be
# scored directly rather than left as the schema stub this cell used to hold.
# ============================================================================
if alert_scores is None or "event_id" not in test.columns:
    event_summary = pd.DataFrame()
    print("Event-level evaluation needs alerting output and Phase 2 event columns.")
else:
    ev = test.dropna(subset=["event_id"]).copy()
    ev = ev[ev["event_peak_pm25"] >= 150]           # hazardous episodes only
    if ev.empty:
        event_summary = pd.DataFrame()
        print("No extreme events in the test period - nothing to score.")
    else:
        alert_flag = pd.Series(predicted_extreme, index=test.index)
        rows = []
        for (loc, eid), grp in ev.groupby(["location_id", "event_id"]):
            start = grp.timestamp_utc.min()
            # Alerts raised at this station in the 48h before onset.
            window = test[(test.location_id == loc) &
                          (test.timestamp_utc < start) &
                          (test.timestamp_utc >= start - pd.Timedelta(hours=48))]
            fired = window.index[alert_flag.reindex(window.index).fillna(0).astype(bool)]
            if len(fired):
                first = test.loc[fired, "timestamp_utc"].min()
                lead = (start - first).total_seconds() / 3600.0
            else:
                lead = np.nan
            rows.append({"location_id": loc, "event_id": eid, "event_start": start,
                         "peak_pm25": grp.event_peak_pm25.iloc[0],
                         "detected": bool(len(fired)), "lead_time_hours": lead})
        event_summary = pd.DataFrame(rows)
        detected = event_summary.detected.mean()
        print(f"Extreme events in test : {len(event_summary)}")
        print(f"Detected before onset  : {event_summary.detected.sum()} ({detected:.0%})")
        if event_summary.detected.any():
            print(f"Median lead time       : {event_summary.lead_time_hours.median():.1f} h")
            print(f"Max lead time          : {event_summary.lead_time_hours.max():.1f} h")
        else:
            print("No event was flagged before it began - zero operational lead time.")
        display(event_summary.head(10))


Extreme events in test : 275
Detected before onset  : 167 (61%)
Median lead time       : 43.0 h
Max lead time          : 48.0 h


,location_id,event_id,event_start,peak_pm25,detected,lead_time_hours
0,5616,444.0,2026-06-13 15:30:00+00:00,165.0,False,NaN
1,5616,458.0,2026-06-21 16:30:00+00:00,162.0,True,34.0
2,6962,245.0,2026-07-31 22:30:00+00:00,218.0,False,NaN
3,6988,380.0,2026-07-30 13:30:00+00:00,199.0,True,45.0
4,6988,425.0,2026-09-06 23:30:00+00:00,160.0,False,NaN
5,8118,393.0,2026-06-17 06:30:00+00:00,227.0,True,28.0
6,8118,399.0,2026-06-21 16:30:00+00:00,162.0,False,NaN
7,234568,533.0,2026-07-30 15:30:00+00:00,158.0,False,NaN
8,234568,536.0,2026-07-31 19:30:00+00:00,157.0,True,28.0
9,234568,566.0,2026-08-13 00:30:00+00:00,159.0,True,4.0


### Result — 🎯 61% of episodes caught, median 43 hours ahead

```
Extreme events in test : 275
Detected before onset  : 167  (61%)
Median lead time       : 43.0 h
Max lead time          : 48.0 h
```

**This is the most operationally meaningful result in the entire pipeline.**

Hourly recall (0.324) and event-level detection (0.61) answer different questions. Hourly recall asks *"was this specific hour flagged?"*; a multi-hour episode can be missed for its first six hours and still be caught in time to act. What matters operationally is: **did we warn before it started, and how early?**

61% of hazardous episodes were flagged before onset, with a **median 43 hours of lead time** — enough to close schools, halt construction, and issue advisories. Compare with Phase 5, where thresholding a regression output gave **zero** extreme recall at 24 h.

That contrast is the design lesson: for alerting, *classify the event* (`target_extreme_within_24h`) rather than *threshold a regression*. The regression hedges toward the mean and never crosses 150; the classifier outputs a probability that can be thresholded at whatever recall the product needs.

Caveats worth keeping: the 39% missed are real misses, precision remains 7%, and this test period is monsoon — the model's weakest season.


## 9. Calibration

Do not expose raw classifier probabilities as confidence until calibrated.

Evaluate:

- reliability curve
- Brier score
- expected calibration error

Then use Platt scaling or isotonic calibration on a validation period.

In [11]:
# ============================================================================
# PROBABILITY CALIBRATION
# ============================================================================
# A RandomForest's predict_proba is a vote share, not a probability: it is
# usually over-confident in the middle and under-confident at the extremes.
# That matters here because the alert threshold is applied to those numbers -
# an uncalibrated 0.30 does not mean "30% chance".
#
# FrozenEstimator wraps the already-fitted classifier so the calibration map is
# learned on the held-out calibration slice WITHOUT refitting it, and the test
# period stays untouched. (Replaces cv="prefit", removed in scikit-learn 1.6.)
#
# GUARD: isotonic calibration cannot learn a mapping for a class it never
# observes. If the calibration slice is missing regimes - common on a short
# window, where severe air simply does not occur in that period - calibration
# is skipped rather than silently fitted on a subset of classes, which would
# renumber the probability columns and corrupt the alert threshold.
# ============================================================================
y_calib = calib["future_regime"].astype(str)
missing_classes = sorted(set(map(str, clf.classes_)) - set(y_calib.unique()))

if len(calib) < 100 or y_calib.nunique() < 2:
    calibrated = None
    print(f"Calibration slice too small ({len(calib)} rows) or single-class - skipped.")
elif missing_classes:
    calibrated = None
    print(f"Calibration slice is missing regimes {missing_classes} - calibration skipped.")
    print("Isotonic calibration cannot map a class it never sees. Widen the "
          "dataset window so every regime appears in the calibration period; "
          "until then the alert threshold stays on RAW probabilities.")
else:
    calibrated = CalibratedClassifierCV(FrozenEstimator(clf), method="isotonic")
    calibrated.fit(imputer.transform(calib[FEATURES]), y_calib)

    raw_probs = clf.predict_proba(Xte_i)
    cal_probs = calibrated.predict_proba(Xte_i)
    print("Calibrated classifier fitted on the calibration slice.")

    if extreme_idx:
        j_raw = extreme_idx[0]
        cal_classes = list(map(str, calibrated.classes_))
        j_cal = cal_classes.index("extreme") if "extreme" in cal_classes else None
        actual = (test[REG_TARGET] >= 150).astype(int).to_numpy()
        if j_cal is not None and 0 < actual.sum() < len(actual):
            from sklearn.metrics import brier_score_loss
            print("\nBrier score for P(extreme) - lower is better:")
            print(f"  raw        : {brier_score_loss(actual, raw_probs[:, j_raw]):.5f}")
            print(f"  calibrated : {brier_score_loss(actual, cal_probs[:, j_cal]):.5f}")
            print("A lower calibrated score means the probabilities are more "
                  "trustworthy AS probabilities - which is what makes a tuned "
                  "alert threshold meaningful rather than arbitrary.")
        else:
            print("Not enough extreme variation in the test period to score calibration.")


Calibrated classifier fitted on the calibration slice.

Brier score for P(extreme) - lower is better:
  raw        : 0.00251
  calibrated : 0.00225
A lower calibrated score means the probabilities are more trustworthy AS probabilities - which is what makes a tuned alert threshold meaningful rather than arbitrary.


### Result — calibration improves probability quality

| | Brier score (lower is better) |
|---|---:|
| raw RandomForest | 0.00251 |
| **isotonic-calibrated** | **0.00225** |

A **10% improvement**. Small in absolute terms because the base rate is tiny, but it matters here specifically because the alert threshold is applied to these probabilities — an uncalibrated 0.10 from a random forest is a vote share, not "a 10% chance".

`FrozenEstimator` fits the calibration map on the held-out calibration slice without refitting the classifier, so the test period stays untouched. (This replaces `cv="prefit"`, removed in scikit-learn 1.6.)

The guard in this cell matters too: isotonic calibration cannot learn a mapping for a class it never observes, so if the calibration slice were missing a regime the cell skips rather than silently fitting on a subset of classes — which would renumber the probability columns and corrupt the threshold.


## 10. Save the regime-aware model bundle

Bundle:

- classifier
- imputer
- expert models
- feature contract
- thresholds
- class weights
- training window
- dataset version
- calibration metadata
- validation metrics

This bundle must not be promoted until extreme-event performance passes the AeroPulse acceptance criteria.

In [12]:
# ============================================================================
# PERSIST THE REGIME-AWARE BUNDLE
# ============================================================================
# Classifier, calibrator, imputer, experts and fallback are saved TOGETHER.
# Loading any one without the others silently produces wrong predictions
# rather than an error - the imputer and feature order are part of the model.
# ============================================================================
REGIME_DIR = ARTIFACT_DIR / "phase6"
REGIME_DIR.mkdir(parents=True, exist_ok=True)

import joblib

bundle = {
    "classifier": clf,
    "calibrated_classifier": calibrated,
    "imputer": imputer,
    "experts": EXPERTS,
    "fallback_model": fallback_model,
    "features": FEATURES,
    "horizon_hours": TARGET_H,
    "regime_thresholds": [-np.inf, 30, 60, 90, 150, np.inf],
    "regime_labels": ["normal", "moderate", "high", "very_high", "extreme"],
    "alert_threshold": alert_threshold,
    "alert_threshold_source": "swept on the calibration slice, F-beta(2)",
    "training_cutoff": str(cut),
    "train_window": [str(train.timestamp_utc.min()), str(train.timestamp_utc.max())],
    "test_window": [str(test.timestamp_utc.min()), str(test.timestamp_utc.max())],
    "moe_metrics": moe_scores,
    "single_model_metrics": single_scores,
    "alert_metrics": alert_scores,
    "version": "2.0.0-regime-aware",
}
joblib.dump(bundle, REGIME_DIR / "pm25_regime_aware_bundle.joblib")

report = {k: v for k, v in bundle.items()
          if k not in ("classifier", "calibrated_classifier", "imputer",
                       "experts", "fallback_model")}
report["experts_trained"] = sorted(EXPERTS.keys())
with open(REGIME_DIR / "phase6_report.json", "w") as f:
    json.dump(report, f, indent=2, default=str)
if not event_summary.empty:
    event_summary.to_csv(REGIME_DIR / "event_lead_time.csv", index=False)

print("Saved:", REGIME_DIR / "pm25_regime_aware_bundle.joblib")
for p in sorted(REGIME_DIR.glob("*")):
    print(f"  {p.name:36} {p.stat().st_size/1e6:7.2f} MB")
print("\nHONEST SUMMARY")
print(f"  MoE RMSE          : {moe_scores['RMSE']:.2f} ug/m3 (R2 {moe_scores['R2']:.3f})")
print(f"  Single model RMSE : {single_scores['RMSE']:.2f} ug/m3")
if alert_scores:
    print(f"  Extreme recall    : {alert_scores['recall']:.1%} "
          f"on {alert_scores['actual_extreme']} actual events")


Saved: /Users/sunil/Documents/hackathon/AeroPulse/AeroPulse/AeroPulse_ML_Notebooks/pm25_estimator/artifacts/pm25/phase6/pm25_regime_aware_bundle.joblib
  event_lead_time.csv                     0.01 MB
  phase6_report.json                      0.00 MB
  pm25_regime_aware_bundle.joblib      1639.93 MB

HONEST SUMMARY
  MoE RMSE          : 19.28 ug/m3 (R2 0.100)
  Single model RMSE : 17.99 ug/m3
  Extreme recall    : 32.4% on 592 actual events


### Result — bundle saved, with one practical warning

```
MoE RMSE          : 19.28 ug/m3 (R2 0.100)
Single model RMSE : 17.99 ug/m3
Extreme recall    : 32.4% on 592 actual events
Event detection   : 61% at 43h median lead time
```

> **⚠️ The bundle is 1.64 GB.** A 250-tree RandomForest fitted on 1.2M rows plus five LightGBM experts does not belong in a latency-sensitive serving path. Before deployment: replace the forest with a LightGBM classifier (far smaller, usually better calibrated), or prune/quantise it. This is a real deployment blocker, not a cosmetic one.

Classifier, calibrator, imputer, experts and fallback are stored **together** deliberately — loading any one without the others produces silently wrong predictions rather than an error, since the imputer and feature order are part of the model.

The report records `alert_threshold_source` ("swept on the calibration slice, F-beta(2)") so a future reader can see the threshold was not hand-picked.


## Phase-6 acceptance criteria

Before deep learning:

1. Regime classifier has useful extreme recall.
2. MoE does not materially degrade normal-regime MAE.
3. High/very-high error is substantially lower than the Phase-5 baseline.
4. Extreme-event recall is materially better than the single LightGBM model.
5. Spatial holdout remains acceptable.
6. Seasonal/event holdout remains acceptable.
7. Probabilities are calibrated.
8. Online features exactly match training semantics.
9. Historical lag features come from the time-series store.
10. Forecast weather is used for future horizons rather than historical weather.